In [4]:
!pip3 install --upgrade --quiet langchain langchain-community langchain-openai chromadb
!pip3 install --upgrade --quiet pypdf pandas streamlit python-dotenv"

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [6]:
import os
import tempfile
import streamlit as st
import pandas as pd 
from dotenv import load_dotenv

In [7]:
%pip install sentence-transformers pdfplumber numpy

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pdfplumber

def extract_raw_text(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + "\n"
    return full_text
text = extract_raw_text("C:\\Users\\DELL\\OneDrive\\Desktop\\Projects\\app dev\\nova\\lib\\core\\ai\\data\\AI compree 25.pdf")
print(text[:2000])

1. Group the given data (185, 72),
(170, 56), (168, 60), (179,68),
(182,72), (188,77) into two groups
using an unsupervised algorithm up
to two iterations using euclidean
distance . Initially choose the first
two objects as initial centroids. [15
marks].
Solution:
Given, number of clusters to be created (K) = 2 say c1 and c2,
number of iterations = 2 and The given data points can be represented in tabular form as:also, first two
objects as initial centroids:Centroid for first cluster c1 = (185, 72)Centroid for second cluster c2 = (170, 56)
Iteration 1: Now calculating similarity by using Euclidean distance measure as: Euclidean distance
calculation
Representing above information in tabular form:
Distance of each data points from cluster centroids
The resulting cluster after first iteration is: C1- 1 4 5 6, C2 2 3
Iteration 2: Now calculating centroid for each cluster
Calculating centroid as mean of data points Now, again calculating similarity:
Distance calculation between data points 

In [9]:
import os

folder = "C:/Users/DELL/OneDrive/Desktop/Projects/app dev/nova/lib/core/ai/data/"
files = os.listdir(folder)
for f in files:
    print(f)

AI compree 25.pdf
AI_CSF407_Compre_Solutions.pdf
CS F407-AI Mid sem 25-26.docx.pdf
KEY PART-A-end-SetA.pdf
KEY PART-B-end.pdf
MidSem_AI_CSF407_Solution.pdf
PostCompre.pdf


In [13]:
path = "C:/Users/DELL/OneDrive/Desktop/Projects/app dev/nova/lib/core/ai/data/AI compree 25.pdf"

print("Reading:", path)
text = extract_raw_text(path)
print(text[:2000])

Reading: C:/Users/DELL/OneDrive/Desktop/Projects/app dev/nova/lib/core/ai/data/AI compree 25.pdf
1. Group the given data (185, 72),
(170, 56), (168, 60), (179,68),
(182,72), (188,77) into two groups
using an unsupervised algorithm up
to two iterations using euclidean
distance . Initially choose the first
two objects as initial centroids. [15
marks].
Solution:
Given, number of clusters to be created (K) = 2 say c1 and c2,
number of iterations = 2 and The given data points can be represented in tabular form as:also, first two
objects as initial centroids:Centroid for first cluster c1 = (185, 72)Centroid for second cluster c2 = (170, 56)
Iteration 1: Now calculating similarity by using Euclidean distance measure as: Euclidean distance
calculation
Representing above information in tabular form:
Distance of each data points from cluster centroids
The resulting cluster after first iteration is: C1- 1 4 5 6, C2 2 3
Iteration 2: Now calculating centroid for each cluster
Calculating centroid as

In [14]:
%pip install groq

Note: you may need to restart the kernel to use updated packages.


In [16]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": "Say hello"
        }
    ]
)

print(response.choices[0].message.content)

Hello. How can I assist you today?


In [17]:
import json

def extract_questions_with_groq(raw_text):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": f"""
Extract all questions from this exam paper text.

For each question return a JSON array where each item has:
- "question_text": the full question only, no solution, no answer
- "marks": marks allocated as integer
- "question_type": "mcq" or "long_answer"

Return ONLY the JSON array, nothing else. No explanation, no markdown.

TEXT:
{raw_text[:6000]}
"""
            }
        ]
    )

    raw = response.choices[0].message.content
    return json.loads(raw)


questions = extract_questions_with_groq(text)
print(json.dumps(questions[:3], indent=2))

[
  {
    "question_text": "Group the given data (185, 72), (170, 56), (168, 60), (179,68), (182,72), (188,77) into two groups using an unsupervised algorithm up to two iterations using euclidean distance . Initially choose the first two objects as initial centroids.",
    "marks": 15,
    "question_type": "long_answer"
  },
  {
    "question_text": "For the given data from a retail company predict the monthly sales revenue for size 2800 sq ft and 12 employees using KNN based on three nearest neighbors using Euclidean distance.",
    "marks": 10,
    "question_type": "long_answer"
  },
  {
    "question_text": "Build a decision tree using the ID3 algorithm for the given training data in the table (Buy Computer data), and predict the class of the following new example: age<=30, income=medium, student=yes, credit-rating=fair.",
    "marks": 20,
    "question_type": "long_answer"
  }
]


In [18]:
def extract_all_questions(raw_text, chunk_size=6000):
    all_questions = []

    chunks = [
        raw_text[i:i + chunk_size]
        for i in range(0, len(raw_text), chunk_size)
    ]

    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i + 1}/{len(chunks)}...")

        try:
            questions = extract_questions_with_groq(chunk)
            all_questions.extend(questions)

        except Exception as e:
            print(f"Chunk {i + 1} failed, skipping")
            print(f"Error: {e}")

    return all_questions


all_questions = extract_all_questions(text)

for q in all_questions:
    q["subject"] = "AI"
    q["year"] = 2025
    q["exam_type"] = "compre"


with open("question_bank.json", "w") as f:
    json.dump(all_questions, f, indent=2)


print(f"Total questions extracted: {len(all_questions)}")

Processing chunk 1/1...
Total questions extracted: 11


In [22]:
print(f"Total questions: {len(all_questions)}")

Total questions: 11


In [23]:
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract question texts
question_texts = [q["question_text"] for q in all_questions]

# Generate embeddings
embeddings = model.encode(question_texts)

# Print shape
print(f"Embeddings shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (11, 384)


In [24]:
import numpy as np

# Save embeddings to file
np.save("embeddings.npy", embeddings)

print("Embeddings saved!")

# You can later load with embeddings = np.load("embeddings.npy")

Embeddings saved!


In [26]:
def mmr(query_embedding, candidate_embeddings, candidate_questions, k=5, alpha=0.7):
    from numpy.linalg import norm
    
    def cosine_sim(a, b):
        return np.dot(a, b) / (norm(a) * norm(b))
    
    # similarity between query and all candidates
    query_sims = [cosine_sim(query_embedding, e) for e in candidate_embeddings]
    
    selected = []
    selected_embeddings = []
    remaining = list(range(len(candidate_questions)))
    
    for _ in range(min(k, len(remaining))):
        mmr_scores = []
        for i in remaining:
            relevance = query_sims[i]
            if selected_embeddings:
                diversity = max(cosine_sim(candidate_embeddings[i], s) for s in selected_embeddings)
            else:
                diversity = 0
            score = alpha * relevance - (1 - alpha) * diversity
            mmr_scores.append((i, score))
        
        best = max(mmr_scores, key=lambda x: x[1])[0]
        selected.append(candidate_questions[best])
        selected_embeddings.append(candidate_embeddings[best])
        remaining.remove(best)
    
    return selected

# test it - use first question as query
query = embeddings[0]
diverse_questions = mmr(query, embeddings, all_questions, k=5)

print(f"Selected {len(diverse_questions)} diverse questions:")
for i, q in enumerate(diverse_questions):
    print(f"\n{i+1}. {q['question_text'][:100]}...")

Selected 5 diverse questions:

1. Group the given data (185, 72), (170, 56), (168, 60), (179,68), (182,72), (188,77) into two groups u...

2. A small bookstore recorded weekly sales (in units) of a new book over 15 weeks: 20, 22, 25, 30, 2, 3...

3. With least square error fit use the given data to predict the value for x1=1, x2=2....

4. For the given data from a retail company predict the monthly sales revenue for size 2800 sq ft and 1...

5. Given the initial state of 4 queens, show the steps for the hill climbing algorithm until it termina...


In [27]:
def generate_question(diverse_examples, subject="AI"):
    
    examples_text = ""
    for i, q in enumerate(diverse_examples):
        examples_text += f"\nExample {i+1} ({q['marks']} marks, {q['question_type']}):\n{q['question_text']}\n"
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""You are an exam question generator for {subject}.
Below are example questions from previous exams at this college.
These set the difficulty level and style standard.

{examples_text}

Generate ONE new original exam question in the same style and difficulty range.
Do not repeat any of the above questions.
Return JSON with:
- "question": the question text
- "marks": suggested marks
- "question_type": "mcq" or "long_answer"
- "topic": the topic it covers

Return ONLY JSON, nothing else. No markdown, no backticks."""
        }]
    )
    
    raw = response.choices[0].message.content
    print("Raw response:", raw)  # see what came back
    
    # strip markdown if present
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    
    return json.loads(raw.strip())

new_question = generate_question(diverse_questions)
print(json.dumps(new_question, indent=2))

Raw response: {
  "question": "A company has recorded the average monthly customer complaints and the average response time for customer service over 8 months: (10, 5), (12, 4), (8, 6), (15, 3), (11, 5), (9, 6), (13, 4), (14, 5). Use a supervised learning algorithm to predict the number of customer complaints for a response time of 4.5 minutes using linear regression. Show the steps to calculate the regression line and the predicted value.",
  "marks": 12,
  "question_type": "long_answer",
  "topic": "Linear Regression"
}
{
  "question": "A company has recorded the average monthly customer complaints and the average response time for customer service over 8 months: (10, 5), (12, 4), (8, 6), (15, 3), (11, 5), (9, 6), (13, 4), (14, 5). Use a supervised learning algorithm to predict the number of customer complaints for a response time of 4.5 minutes using linear regression. Show the steps to calculate the regression line and the predicted value.",
  "marks": 12,
  "question_type": "long_

In [28]:
all_pdfs = [
    ("AI compree 25.pdf", 2025, "compre"),
    ("CS F407-AI Mid sem 25-26.docx.pdf", 2026, "midsem"),
    ("KEY PART-A-end-SetA.pdf", 2024, "compre"),
    ("KEY PART-B-end.pdf", 2024, "compre"),
]

folder = "C:/Users/DELL/OneDrive/Desktop/Projects/app dev/nova/lib/core/ai/data/"

for filename, year, exam_type in all_pdfs:
    print(f"\nProcessing {filename}...")
    t = extract_raw_text(folder + filename)
    qs = extract_all_questions(t)
    for q in qs:
        q["subject"] = "AI"
        q["year"] = year
        q["exam_type"] = exam_type
    all_questions.extend(qs)

# save updated bank
with open("question_bank.json", "w") as f:
    json.dump(all_questions, f, indent=2)

print(f"\nTotal questions in bank: {len(all_questions)}")


Processing AI compree 25.pdf...
Processing chunk 1/1...

Processing CS F407-AI Mid sem 25-26.docx.pdf...
Processing chunk 1/2...
Processing chunk 2/2...

Processing KEY PART-A-end-SetA.pdf...
Processing chunk 1/1...

Processing KEY PART-B-end.pdf...
Processing chunk 1/2...
Processing chunk 2/2...

Total questions in bank: 65


In [29]:
# re-embed all questions
question_texts = [q["question_text"] for q in all_questions]
embeddings = model.encode(question_texts)

# save
np.save("embeddings.npy", embeddings)

print(f"Embeddings shape: {embeddings.shape}")

Embeddings shape: (65, 384)
